# Cuaderno de Trabajo: Estrategia de Trading Algoritmico

Bienvenido a tu cuaderno de trabajo (**Jupyter Notebook**) para el diseno, analisis cuantitativo y validacion de estrategias de trading.

### Contenido del Cuaderno:
1. **Configuracion del Entorno:** Conexion con librerias y proyectos adyacentes (`robot`).
2. **Carga de Datos de Mercado:** Conexion con **MetaTrader 5 (MT5)** o generacion de series historicas simuladas.
3. **Calculo de Indicadores Tecnicos:** Medias moviles (EMA 200), volatilidad (ATR 14) y rangos de sesion horaria (Asia / Londres / NY).
4. **Visualizacion y Graficos:** Inspeccion visual de velas y niveles tecnicos.
5. **Implementacion de Estrategia:** Reglas de entrada, Stop Loss dinamico y Take Profit asimetrico.
6. **Motor de Backtesting:** Simulacion historica con metricas de rendimiento (Win Rate, Profit Factor, Balance).
7. **Curva de Capital:** Evolucion del balance a lo largo del tiempo.

## 1. Configuracion del Entorno e Importaciones

In [ ]:
import sys
import os
from pathlib import Path
from datetime import datetime, timedelta, timezone
import numpy as np
import pandas as pd

# 1. Configuracion de rutas (anade este directorio y el directorio del robot si existe)
current_dir = Path.cwd()
robot_dir = current_dir.parent / "robot"

for p in [current_dir, robot_dir]:
    if p.exists() and str(p) not in sys.path:
        sys.path.append(str(p))

# 2. Verificacion de librerias visuales y de conexion
try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    print("[INFO] 'matplotlib' no esta instalado. Para ver graficos ejecuta 'pip install matplotlib'.")

try:
    import MetaTrader5 as mt5
    HAS_MT5 = True
except ImportError:
    HAS_MT5 = False
    print("[INFO] 'MetaTrader5' no esta instalado o no disponible en este sistema.")

print("\n--- Estado del Entorno ---")
print(f"Directorio actual: {current_dir}")
print(f"Proyecto robot adyacente detectado: {robot_dir.exists()}")
print(f"Matplotlib (graficos): {'Disponible [OK]' if HAS_MATPLOTLIB else 'No disponible [AVISO]'}")
print(f"MetaTrader 5: {'Disponible [OK]' if HAS_MT5 else 'No disponible [AVISO]'}")

## 2. Carga y Preparacion de Datos de Mercado

Puedes descargar datos en tiempo real o historicos de **MetaTrader 5** (ej. `XAUUSD` o `EURUSD`) o utilizar el simulador integrado para hacer pruebas inmediatas.

In [ ]:
def cargar_datos_mercado(simbolo="XAUUSD", n_barras=3000, dias_simulados=40):
    """
    Obtiene datos de MT5 si esta activo, o genera una serie OHLCV sintetica realista.
    """
    df = None
    
    # Intentar obtener datos desde MT5
    if HAS_MT5:
        try:
            if mt5.initialize():
                rates = mt5.copy_rates_from_pos(simbolo, mt5.TIMEFRAME_M5, 0, n_barras)
                mt5.shutdown()
                if rates is not None and len(rates) > 0:
                    df = pd.DataFrame(rates)
                    df["time"] = pd.to_datetime(df["time"], unit="s")
                    print(f"[OK] Se cargaron {len(df)} velas de {simbolo} desde MetaTrader 5.")
        except Exception as e:
            print(f"No se pudo conectar a MT5 ({e}). Se usaran datos de simulacion.")
    
    # Generacion de datos sinteticos si MT5 no esta disponible o no devolvio velas
    if df is None or df.empty:
        print(f"[SIM] Generando {dias_simulados} dias de datos simulados para {simbolo} (M5)...")
        np.random.seed(42)
        paso_min = 5
        total_velas = (dias_simulados * 24 * 60) // paso_min
        fecha_inicio = datetime.now(timezone.utc) - timedelta(days=dias_simulados)
        tiempos = [fecha_inicio + timedelta(minutes=i * paso_min) for i in range(total_velas)]
        
        precio_inicial = 2650.0  # Referencia para XAUUSD
        retornos = np.random.normal(0.000015, 0.00075, total_velas)
        cierres = precio_inicial * np.exp(np.cumsum(retornos))
        
        volatilidad = cierres * 0.0006
        altos = cierres + np.abs(np.random.normal(0, volatilidad))
        bajos = cierres - np.abs(np.random.normal(0, volatilidad))
        aperturas = cierres + np.random.normal(0, volatilidad * 0.4)
        
        # Consistencia de velas japonesas OHLC
        altos = np.maximum(altos, np.maximum(aperturas, cierres))
        bajos = np.minimum(bajos, np.minimum(aperturas, cierres))
        volumen = np.random.randint(50, 1200, total_velas)
        
        df = pd.DataFrame({
            "time": tiempos,
            "open": aperturas,
            "high": altos,
            "low": bajos,
            "close": cierres,
            "tick_volume": volumen
        })
        print(f"[OK] {len(df)} velas simuladas generadas.")
        
    return df

df_mercado = cargar_datos_mercado("XAUUSD", n_barras=2500, dias_simulados=30)
print(df_mercado.tail())


## 3. Indicadores Tecnicos y Estructura de Mercado

Calculamos:
- **EMA 200 (Tendencia Macro):** Sesgo principal de mercado.
- **ATR 14 (Volatilidad):** Para calculo dinamico de Stop Loss.
- **Rango de Sesion Asiatica (00:00 - 07:00 UTC):** Maximo y minimo como niveles de ruptura para Londres y Nueva York.

In [ ]:
def calcular_indicadores(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    
    # 1. EMA 200
    data["ema_200"] = data["close"].ewm(span=200, adjust=False).mean()
    
    # 2. Average True Range (ATR 14)
    prev_close = data["close"].shift(1)
    tr1 = data["high"] - data["low"]
    tr2 = (data["high"] - prev_close).abs()
    tr3 = (data["low"] - prev_close).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    data["atr_14"] = tr.rolling(window=14).mean()
    
    # 3. Rango de sesion (00:00 a 07:00 UTC)
    data["hour"] = data["time"].dt.hour
    data["date"] = data["time"].dt.date
    
    mascara_asia = (data["hour"] >= 0) & (data["hour"] < 7)
    velas_asia = data[mascara_asia]
    
    highs_dia = velas_asia.groupby("date")["high"].max()
    lows_dia = velas_asia.groupby("date")["low"].min()
    
    data["session_high"] = data["date"].map(highs_dia)
    data["session_low"] = data["date"].map(lows_dia)
    
    return data

df_analisis = calcular_indicadores(df_mercado)
print("[OK] Indicadores calculados exitosamente.")
print(df_analisis[["time", "close", "ema_200", "atr_14", "session_high", "session_low"]].dropna().tail())

## 4. Visualizacion Grafica

In [ ]:
if HAS_MATPLOTLIB:
    muestra = df_analisis.dropna().tail(150)
    
    plt.figure(figsize=(14, 6))
    plt.plot(muestra["time"], muestra["close"], label="Precio de Cierre", color="#1f77b4", alpha=0.85)
    plt.plot(muestra["time"], muestra["ema_200"], label="EMA 200 (Tendencia)", color="#ff7f0e", linewidth=1.6)
    
    if not muestra["session_high"].isna().all():
        plt.plot(muestra["time"], muestra["session_high"], label="Maximo Sesion", color="green", linestyle="--", alpha=0.7)
        plt.plot(muestra["time"], muestra["session_low"], label="Minimo Sesion", color="red", linestyle="--", alpha=0.7)
    
    plt.title("Analisis Tecnico: Precio, EMA 200 y Niveles de Sesion", fontsize=13)
    plt.xlabel("Fecha y Hora")
    plt.ylabel("Precio")
    plt.legend(loc="upper left")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.xticks(rotation=35)
    plt.tight_layout()
    plt.show()
else:
    print("[INFO] Graficos omitidos porque matplotlib no esta instalado.")

## 5. Logica de Estrategia y Generacion de Senales

Definicion de las reglas de disparo:
- **BUY:** Quiebre por encima del maximo asiatico + Precio > EMA 200.
- **SELL:** Quiebre por debajo del minimo asiatico + Precio < EMA 200.
- **Riesgo:** Stop Loss dinamico ($1.5 \times \text{ATR}$) y Take Profit $1:2$.

In [ ]:
class EstrategiaSessionBreakout:
    def __init__(self, mult_sl=1.5, ratio_tp=2.0):
        self.mult_sl = mult_sl
        self.ratio_tp = ratio_tp
        
    def detectar_senales(self, df: pd.DataFrame) -> pd.DataFrame:
        senales = []
        
        for i in range(1, len(df)):
            prev = df.iloc[i - 1]
            curr = df.iloc[i]
            
            # Validar que los indicadores esten disponibles
            if pd.isna(curr["ema_200"]) or pd.isna(curr["atr_14"]) or pd.isna(curr["session_high"]):
                continue
                
            # Horario operativo (07:00 a 15:00 UTC)
            hora = curr["time"].hour if hasattr(curr["time"], "hour") else 0
            if not (7 <= hora <= 15):
                continue
                
            precio = curr["close"]
            atr = curr["atr_14"]
            dist_sl = atr * self.mult_sl
            
            # Senal de COMPRA
            if prev["close"] <= prev["session_high"] and precio > curr["session_high"] and precio > curr["ema_200"]:
                senales.append({
                    "bar_index": i,
                    "time": curr["time"],
                    "type": "BUY",
                    "entry": precio,
                    "sl": precio - dist_sl,
                    "tp": precio + (dist_sl * self.ratio_tp),
                    "atr": atr
                })
            # Senal de VENTA
            elif prev["close"] >= prev["session_low"] and precio < curr["session_low"] and precio < curr["ema_200"]:
                senales.append({
                    "bar_index": i,
                    "time": curr["time"],
                    "type": "SELL",
                    "entry": precio,
                    "sl": precio + dist_sl,
                    "tp": precio - (dist_sl * self.ratio_tp),
                    "atr": atr
                })
                
        return pd.DataFrame(senales)

estrategia = EstrategiaSessionBreakout()
df_senales = estrategia.detectar_senales(df_analisis)
print(f"Senales identificadas en el periodo: {len(df_senales)}")
if not df_senales.empty:
    print(df_senales.head())


## 6. Backtesting y Reporte de Rendimiento

In [ ]:
def simular_backtest(df: pd.DataFrame, df_senales: pd.DataFrame, capital_inicial=10000.0, riesgo_pct=0.01):
    balance = capital_inicial
    trades = []
    equity = [{"time": df["time"].iloc[0], "balance": balance}]
    
    if df_senales.empty:
        return pd.DataFrame(), pd.DataFrame(equity)
        
    senales_map = {row["bar_index"]: row for _, row in df_senales.iterrows()}
    posicion = None
    
    for i in range(len(df)):
        row = df.iloc[i]
        t = row["time"]
        
        # Evaluar salida de posicion existente
        if posicion is not None:
            pnl = None
            motivo = None
            
            if posicion["type"] == "BUY":
                if row["low"] <= posicion["sl"]:
                    pnl = -posicion["riesgo_usd"]
                    motivo = "Stop Loss"
                elif row["high"] >= posicion["tp"]:
                    pnl = posicion["riesgo_usd"] * posicion["ratio"]
                    motivo = "Take Profit"
            elif posicion["type"] == "SELL":
                if row["high"] >= posicion["sl"]:
                    pnl = -posicion["riesgo_usd"]
                    motivo = "Stop Loss"
                elif row["low"] <= posicion["tp"]:
                    pnl = posicion["riesgo_usd"] * posicion["ratio"]
                    motivo = "Take Profit"
                    
            if pnl is not None:
                balance += pnl
                trades.append({
                    "open_time": posicion["time"],
                    "close_time": t,
                    "type": posicion["type"],
                    "entry": posicion["entry"],
                    "pnl": pnl,
                    "reason": motivo,
                    "balance": balance
                })
                equity.append({"time": t, "balance": balance})
                posicion = None
                
        # Apertura de posicion si no hay ninguna abierta
        if posicion is None and i in senales_map:
            sig = senales_map[i]
            riesgo_usd = balance * riesgo_pct
            posicion = {
                "time": sig["time"],
                "type": sig["type"],
                "entry": sig["entry"],
                "sl": sig["sl"],
                "tp": sig["tp"],
                "ratio": 2.0,
                "riesgo_usd": riesgo_usd
            }
            
    return pd.DataFrame(trades), pd.DataFrame(equity)

df_trades, df_equity = simular_backtest(df_analisis, df_senales, capital_inicial=10000.0, riesgo_pct=0.01)

if not df_trades.empty:
    ganadas = df_trades[df_trades["pnl"] > 0]
    perdidas = df_trades[df_trades["pnl"] < 0]
    win_rate = (len(ganadas) / len(df_trades)) * 100
    pnl_total = df_trades["pnl"].sum()
    profit_factor = (ganadas["pnl"].sum() / abs(perdidas["pnl"].sum())) if not perdidas.empty and perdidas["pnl"].sum() != 0 else 0
    
    print("=" * 46)
    print("       INFORME DE RENDIMIENTO ESTRATEGIA")
    print("=" * 46)
    print(f"Operaciones Totales:    {len(df_trades)}")
    print(f"Operaciones Ganadoras:  {len(ganadas)} ({win_rate:.1f}%)")
    print(f"Operaciones Perdedoras: {len(perdidas)} ({100 - win_rate:.1f}%)")
    print(f"Beneficio Neto:         ${pnl_total:,.2f}")
    print(f"Profit Factor:          {profit_factor:.2f}")
    print(f"Balance Final:          ${df_trades['balance'].iloc[-1]:,.2f}")
    print("=" * 46)
else:
    print("No se completaron operaciones en el conjunto de datos.")

## 7. Curva de Capital (Equity Curve)

In [ ]:
if HAS_MATPLOTLIB and not df_equity.empty:
    plt.figure(figsize=(12, 5))
    plt.plot(df_equity["time"], df_equity["balance"], label="Capital ($)", color="#2ca02c", linewidth=2)
    plt.axhline(10000, color="gray", linestyle="--", label="Capital Inicial ($10,000)")
    plt.title("Curva de Crecimiento de Capital", fontsize=13)
    plt.xlabel("Fecha")
    plt.ylabel("Balance ($)")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend(loc="upper left")
    plt.tight_layout()
    plt.show()
elif not df_equity.empty:
    print("Evolucion final de equidad:")
    print(df_equity.tail())